### Implementing simple chatbot using Langgraph

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END

## Reducers
from typing import Annotated
from langgraph.graph.message import add_messages


In [ ]:
class State(TypedDict):
    messages:Annotated[list, add_messages]

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="qwen/qwen3-32b")
ans = llm.invoke("Hello, how are you?")
ans.content

### We will start creating nodes

In [ ]:
def superbot(state:State):
    return {"messages":[llm.invoke(state['messages'])]}

In [ ]:
graph = StateGraph(State)

## node
graph.add_node("Superbot", superbot)

## edges
graph.add_edge(START, "Superbot")
graph.add_edge("Superbot", END)

## compile
graph_builder = graph.compile()


In [ ]:
from IPython.display import display,Image
display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
## Invocation

graph_builder.invoke({"messages":"Hello, how are you?"})

### Stream the responses

In [ ]:
for event in graph_builder.stream({"messages":"Hello, I am Adarsh?"}):
    print(event)